# MFE 230X — High-Frequency Finance — Homework 1

**NASDAQ ITCH (TotalView) + EBS/FX high-frequency order & transaction data**

Instructors: Terrence Hendershott, Dmitry Livdan · Fall 2026

**Authors:** Alexandre Marthan · Elias Roubache · Jean Jacob · Matthieu Pascal · Elouan Bahri

---

### Instrument choices
| | Symbol | Description | Why |
|---|---|---|---|
| **Large cap** | **AAPL** | Apple Inc. (~\$1.3T mkt cap on the sample date) | Deepest, most liquid NASDAQ name |
| **Small cap** | **GPRO** | GoPro Inc. (~\$0.7B) | Small, less liquid — sharp contrast in spread / depth / price impact |
| **FX** | **EUR/USD, USD/JPY, EUR/JPY** | The triangle for part (l) | `EUR/JPY ≈ EUR/USD × USD/JPY` |

**Trading day:** `2019-12-30` (09:30–16:00 ET). This is the smallest full-day public NASDAQ
ITCH sample (3.5 GB gzip). Change `DATE`/tickers in the config cell to use any other day.

### Data sources
- **ITCH:** NASDAQ TotalView-ITCH 5.0 daily sample — `https://emi.nasdaq.com/ITCH/Nasdaq%20ITCH/`
  (public FTP/HTTPS mirror of the feed described at nasdaqtrader.com TotalView). The notebook
  downloads, gunzips-on-the-fly, and parses the binary feed, keeping only our two symbols.
- **FX:** **EBS Data Mine** (`ebs.com` / CME) is a *paid* subscription — there is no free
  programmatic download. The `load_fx()` function therefore (1) reads local **EBS Data Mine**
  export files if you point it at them (e.g. the files provided in Datalore), and otherwise
  (2) falls back to **Dukascopy** free historical tick data for the same three pairs so every
  part of the assignment still runs end-to-end.

### Parts implemented
(a) \$ volume/min · (b) #trades & #orders/min · (c) OHLC/min · (d) VWAP/min ·
(e) BBO spread & depth/min · (f) depth within 2×avg spread/min · (g) 5-sec price impact/min ·
(h) 1-sec & 1-min midquote/trade price series · (i) log-returns · (j) realized variance ·
(k) return autocorrelation (Box-Pierce & PACF) · (l) triangular arbitrage in the FX triangle.


## 0 · Configuration

In [ ]:
# ======================= CONFIG — edit here =======================
from pathlib import Path

DATE        = "2019-12-30"          # trading day (YYYY-MM-DD)
ITCH_DATE   = "12302019"            # same day in the ITCH file-name convention (MMDDYYYY)
STOCKS      = ["AAPL", "GPRO"]      # large cap, small cap
FX_PAIRS    = ["EURUSD", "USDJPY", "EURJPY"]

SESSION_START = "09:30:00"          # regular NASDAQ session
SESSION_END   = "16:00:00"
TZ            = "America/New_York"

DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
ITCH_URL = f"https://emi.nasdaq.com/ITCH/Nasdaq%20ITCH/{ITCH_DATE}.NASDAQ_ITCH50.gz"
ITCH_GZ  = DATA_DIR / f"{ITCH_DATE}.NASDAQ_ITCH50.gz"

# FX source: point EBS_DIR at your EBS Data Mine export folder to use it; else Dukascopy is used.
EBS_DIR       = None               # e.g. Path("data/ebs")  — see load_fx() for the expected schema
USE_DUKASCOPY = True               # free fallback if no EBS files found

# The full ITCH file is ~3.5 GB gzip / ~13 GB raw. Parsing a full day for two symbols
# takes ~15–40 min in Datalore. Set to a byte cap for a quick smoke-test, or None for full day.
ITCH_MAX_BYTES = None              # e.g. 2_000_000_000 for a partial-day test
print("Config loaded. ITCH file:", ITCH_URL)


## 1 · Imports

In [ ]:
import os, io, gzip, struct, lzma, urllib.request, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf, pacf

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (11, 4.2); plt.rcParams["figure.dpi"] = 110
SESSION = (pd.Timestamp(f"{DATE} {SESSION_START}"), pd.Timestamp(f"{DATE} {SESSION_END}"))
print("Imports OK")


## 2 · Download the ITCH file

Streams the gzip to disk with HTTP resume so a dropped connection can be retried. Skipped if the
file already exists (e.g. already present in your Datalore project).

In [ ]:
def download_itch(url=ITCH_URL, dest=ITCH_GZ):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 10_000_000:
        print(f"ITCH file already present: {dest} ({dest.stat().st_size/1e9:.2f} GB)")
        return dest
    print("Downloading", url, "->", dest)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(dest, "wb") as f:
        total = int(r.headers.get("Content-Length", 0)); got = 0; chunk = 1 << 20
        while True:
            b = r.read(chunk)
            if not b: break
            f.write(b); got += len(b)
            if total: print(f"\r  {got/1e9:5.2f}/{total/1e9:5.2f} GB", end="")
    print("\nDone.")
    return dest

# download_itch()   # uncomment to fetch; large download


## 3 · ITCH 5.0 parser + limit-order-book reconstruction

The feed is a single chronological stream of every NASDAQ security for the day, wrapped in the
**BinaryFILE** framing (each message is preceded by a 2-byte big-endian length). We stream-gunzip
it and keep only messages for our two `stock_locate` codes (resolved from the *Stock Directory*
`R` messages — never hard-coded).

For each stock we reconstruct:
* **Trades** — from `E` (executed), `C` (executed w/ price), and `P` (non-cross hidden) messages,
  with a **signed trade direction** (+1 buyer-initiated / −1 seller-initiated). For lit executions
  the sign is taken from the *resting* order's side (a resting **sell** limit being executed ⇒ an
  incoming **buy** ⇒ +1; a resting **buy** ⇒ −1).
* **BBO** — by maintaining the displayed book (`A`/`F` add, `E`/`C`/`X` reduce, `D` delete,
  `U` replace) and snapshotting the best bid/ask (price & size) whenever the top of book changes.
* **Order count** — number of new displayed orders (`A`/`F`).

Prices are integers in units of 1/10000 dollar.

In [ ]:
# ITCH 5.0 message body lengths (bytes AFTER the 1-char type), per the TotalView-ITCH 5.0 spec.
MSG_LEN = {'S':11,'R':38,'H':24,'Y':19,'L':25,'V':34,'W':11,'K':27,'J':34,'h':20,
           'A':35,'F':39,'E':30,'C':35,'X':22,'D':18,'U':34,
           'P':43,'Q':39,'B':18,'I':49,'N':19,'O':47,'r':38}
NS_PER_DAY = 24*3600*1_000_000_000

def _ts6(b):  # 6-byte big-endian nanoseconds-since-midnight
    return int.from_bytes(b, "big")

def parse_itch(symbols, gz_path=ITCH_GZ, max_bytes=ITCH_MAX_BYTES):
    """Return {symbol: {'trades':DataFrame,'bbo':DataFrame,'n_orders':int}}."""
    want = set(symbols)
    locate_of = {}                       # symbol -> locate  (filled from R messages)
    sym_of    = {}                       # locate -> symbol  (only our targets)
    books = {}                           # symbol -> {'bid':{px:sz}, 'ask':{px:sz}}
    orders = {}                          # order_ref -> (symbol, side, px, sz)
    trades = defaultdict(list)           # symbol -> list of (ts, price, shares, sign)
    bbo    = defaultdict(list)           # symbol -> list of (ts, bid, ask, bidsz, asksz)
    n_add  = defaultdict(int)

    def best(book_side, is_bid):
        if not book_side: return (np.nan, 0)
        px = max(book_side) if is_bid else min(book_side)
        return (px, book_side[px])

    def snap(sym, ts):
        b = books[sym]
        bpx, bsz = best(b['bid'], True); apx, asz = best(b['ask'], False)
        bbo[sym].append((ts, bpx, apx, bsz, asz))

    date0 = pd.Timestamp(DATE)
    raw = open(gz_path, "rb")
    f = gzip.GzipFile(fileobj=raw)
    buf = b""; pos = 0; nbytes = 0; nmsg = 0
    CH = 1 << 22
    while True:
        block = f.read(CH)
        if not block:
            break
        buf += block; nbytes += len(block)
        i = 0; L = len(buf)
        while i + 2 <= L:
            mlen = (buf[i] << 8) | buf[i+1]
            if mlen == 0 or i + 2 + mlen > L:
                break
            m = buf[i+2:i+2+mlen]; i += 2 + mlen; nmsg += 1
            t = m[0]

            if t == 0x52:  # 'R' Stock Directory
                sym = m[11:19].decode('ascii').strip()
                if sym in want:
                    loc = (m[1] << 8) | m[2]
                    locate_of[sym] = loc; sym_of[loc] = sym
                    books[sym] = {'bid': {}, 'ask': {}}
                continue

            loc = (m[1] << 8) | m[2]
            if loc not in sym_of:
                continue
            sym = sym_of[loc]; ts = _ts6(m[5:11])

            if t == 0x41 or t == 0x46:      # 'A'/'F' Add Order (+MPID)
                oref = int.from_bytes(m[11:19], "big"); side = m[19:20]
                shares = struct.unpack('>I', m[20:24])[0]
                px = struct.unpack('>I', m[32:36])[0]
                orders[oref] = [sym, side, px, shares]
                bk = books[sym]['bid' if side == b'B' else 'ask']
                bk[px] = bk.get(px, 0) + shares
                n_add[sym] += 1; snap(sym, ts)

            elif t == 0x45:                 # 'E' Order Executed
                oref = int.from_bytes(m[11:19], "big")
                ex = struct.unpack('>I', m[19:23])[0]
                o = orders.get(oref)
                if o:
                    sym2, side, px, sz = o
                    sign = +1 if side == b'S' else -1
                    trades[sym2].append((ts, px, ex, sign))
                    o[3] -= ex
                    bk = books[sym2]['bid' if side == b'B' else 'ask']
                    if px in bk:
                        bk[px] -= ex
                        if bk[px] <= 0: del bk[px]
                    if o[3] <= 0: orders.pop(oref, None)
                    snap(sym2, ts)

            elif t == 0x43:                 # 'C' Order Executed With Price
                oref = int.from_bytes(m[11:19], "big")
                ex = struct.unpack('>I', m[19:23])[0]
                # order_ref@11 shares@19 match@23(8) printable@31 exec_price@32(4)
                printable = m[31:32]; epx = struct.unpack('>I', m[32:36])[0]
                o = orders.get(oref)
                if o:
                    sym2, side, px, sz = o
                    if printable == b'Y':
                        sign = +1 if side == b'S' else -1
                        trades[sym2].append((ts, epx, ex, sign))
                    o[3] -= ex
                    bk = books[sym2]['bid' if side == b'B' else 'ask']
                    if px in bk:
                        bk[px] -= ex
                        if bk[px] <= 0: del bk[px]
                    if o[3] <= 0: orders.pop(oref, None)
                    snap(sym2, ts)

            elif t == 0x58:                 # 'X' Order Cancel
                oref = int.from_bytes(m[11:19], "big")
                cx = struct.unpack('>I', m[19:23])[0]
                o = orders.get(oref)
                if o:
                    sym2, side, px, sz = o; o[3] -= cx
                    bk = books[sym2]['bid' if side == b'B' else 'ask']
                    if px in bk:
                        bk[px] -= cx
                        if bk[px] <= 0: del bk[px]
                    snap(sym2, ts)

            elif t == 0x44:                 # 'D' Order Delete
                oref = int.from_bytes(m[11:19], "big")
                o = orders.pop(oref, None)
                if o:
                    sym2, side, px, sz = o
                    bk = books[sym2]['bid' if side == b'B' else 'ask']
                    if px in bk:
                        bk[px] -= sz
                        if bk[px] <= 0: del bk[px]
                    snap(sym2, ts)

            elif t == 0x55:                 # 'U' Order Replace
                oref = int.from_bytes(m[11:19], "big")
                nref = int.from_bytes(m[19:27], "big")
                nsz = struct.unpack('>I', m[27:31])[0]
                npx = struct.unpack('>I', m[31:35])[0]
                o = orders.pop(oref, None)
                if o:
                    sym2, side, px, sz = o
                    bk = books[sym2]['bid' if side == b'B' else 'ask']
                    if px in bk:
                        bk[px] -= sz
                        if bk[px] <= 0: del bk[px]
                    orders[nref] = [sym2, side, npx, nsz]
                    bk2 = books[sym2]['bid' if side == b'B' else 'ask']
                    bk2[npx] = bk2.get(npx, 0) + nsz
                    snap(sym2, ts)

            elif t == 0x50:                 # 'P' Trade (non-cross, hidden)
                side = m[19:20]; shares = struct.unpack('>I', m[20:24])[0]
                px = struct.unpack('>I', m[32:36])[0]
                sign = +1 if side == b'B' else -1
                trades[sym].append((ts, px, shares, sign))

            # 'Q' cross trades (opening/closing auctions) are excluded from continuous stats.

        buf = buf[i:]
        if max_bytes and nbytes >= max_bytes:
            print(f"  stopped early at {nbytes/1e9:.2f} GB (ITCH_MAX_BYTES)"); break
    f.close(); raw.close()
    print(f"  parsed {nmsg:,} messages; locates: {locate_of}")

    out = {}
    for sym in symbols:
        tr = pd.DataFrame(trades[sym], columns=["ns", "price4", "shares", "sign"])
        qt = pd.DataFrame(bbo[sym], columns=["ns", "bid4", "ask4", "bidsz", "asksz"])
        for df in (tr, qt):
            if len(df):
                df["ts"] = date0 + pd.to_timedelta(df["ns"], unit="ns")
                df.set_index("ts", inplace=True)
        if len(tr):
            tr["price"] = tr["price4"] / 1e4
            tr["dollar"] = tr["price"] * tr["shares"]
        if len(qt):
            qt["bid"] = qt["bid4"] / 1e4; qt["ask"] = qt["ask4"] / 1e4
            qt["mid"] = (qt["bid"] + qt["ask"]) / 2
            qt["spread"] = qt["ask"] - qt["bid"]
            qt = qt[(qt["bid"] > 0) & (qt["ask"] > 0) & (qt["spread"] >= 0)]
        # clip to the regular session
        tr = tr.loc[(tr.index >= SESSION[0]) & (tr.index <= SESSION[1])] if len(tr) else tr
        qt = qt.loc[(qt.index >= SESSION[0]) & (qt.index <= SESSION[1])] if len(qt) else qt
        out[sym] = {"trades": tr, "bbo": qt, "n_orders": n_add[sym]}
        print(f"  {sym}: {len(tr):,} trades, {len(qt):,} quote updates, {n_add[sym]:,} orders added")
    return out


Run the parser (downloads the file first if needed). This is the slow cell for a full day.

In [ ]:
# download_itch()                     # ensure the .gz is present
# STOCK_DATA = parse_itch(STOCKS)     # {'AAPL': {...}, 'GPRO': {...}}


## 4 · FX loader — EBS Data Mine (preferred) or Dukascopy (free fallback)

**EBS Data Mine** is the assignment's intended source but is a paid CME product. If you have the
export files (e.g. provided in Datalore), set `EBS_DIR` and adjust `read_ebs_file()`'s column map
to your product's schema (Level-1 quote files typically carry a date, a millisecond timestamp, the
currency pair, and best bid/ask, sometimes with sizes). Otherwise the notebook downloads
**Dukascopy** free historical ticks for the same three pairs and the same day.

Both paths return, per pair, a tick `DataFrame` indexed by timestamp with `bid, ask, mid`
(and `bidsz, asksz` where available).

In [ ]:
DUKA_POINT = {"EURUSD": 1e5, "EURJPY": 1e3, "USDJPY": 1e3}  # price scaling by quote convention

def read_ebs_file(path, pair):
    """Adapt this to your EBS Data Mine schema. Expected result: ts, bid, ask[, bidsz, asksz]."""
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    def pick(*names):
        for n in names:
            if n in cols: return cols[n]
        return None
    tcol = pick("timestamp", "datetime", "time", "date_time")
    bcol = pick("bid", "bidprice", "best_bid")
    acol = pick("ask", "askprice", "best_ask", "offer")
    if tcol is None or bcol is None or acol is None:
        raise ValueError(f"Map EBS columns for {pair}; found {list(df.columns)}")
    out = pd.DataFrame({"ts": pd.to_datetime(df[tcol]),
                        "bid": df[bcol].astype(float),
                        "ask": df[acol].astype(float)})
    for szname, tgt in [("bidsize", "bidsz"), ("asksize", "asksz")]:
        c = pick(szname, tgt)
        if c: out[tgt] = df[c].astype(float)
    return out.set_index("ts").sort_index()

def download_dukascopy_hour(pair, day, hour):
    y, m, d = day.year, day.month - 1, day.day   # Dukascopy months are 0-indexed
    url = (f"https://datafeed.dukascopy.com/datafeed/{pair}/"
           f"{y:04d}/{m:02d}/{d:02d}/{hour:02d}h_ticks.bi5")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        raw = urllib.request.urlopen(req, timeout=60).read()
    except Exception:
        return None
    if not raw or raw[:5] == b"<html":
        return None
    try:
        dec = lzma.decompress(raw)
    except Exception:
        try: dec = lzma.decompress(raw, format=lzma.FORMAT_ALONE)
        except Exception: return None
    n = len(dec) // 20
    if n == 0: return None
    arr = np.frombuffer(dec[:n*20], dtype=">i4,>i4,>i4,>f4,>f4")
    pt = DUKA_POINT[pair]
    base = pd.Timestamp(f"{day.date()} {hour:02d}:00:00", tz="UTC")
    ts = base + pd.to_timedelta(arr["f0"].astype("int64"), unit="ms")
    return pd.DataFrame({"ask": arr["f1"] / pt, "bid": arr["f2"] / pt,
                         "asksz": arr["f3"], "bidsz": arr["f4"]}, index=ts)

def load_fx_dukascopy(pair, date=DATE):
    day = pd.Timestamp(date)
    parts = [download_dukascopy_hour(pair, day, h) for h in range(24)]
    parts = [p for p in parts if p is not None]
    if not parts:
        raise RuntimeError(f"No Dukascopy data for {pair} on {date} (site may be rate-limiting).")
    df = pd.concat(parts).sort_index()
    df.index = df.index.tz_convert(TZ).tz_localize(None)      # -> exchange local time
    df["mid"] = (df["bid"] + df["ask"]) / 2
    return df

def load_fx(pairs=FX_PAIRS):
    data = {}
    for p in pairs:
        if EBS_DIR is not None:
            f = list(Path(EBS_DIR).glob(f"*{p}*"))
            if f:
                df = read_ebs_file(f[0], p); df["mid"] = (df["bid"] + df["ask"]) / 2
                data[p] = df; print(f"  {p}: EBS {len(df):,} ticks"); continue
        if USE_DUKASCOPY:
            df = load_fx_dukascopy(p); data[p] = df
            print(f"  {p}: Dukascopy {len(df):,} ticks")
    return data

# FX = load_fx()


## 5 · Per-minute descriptive statistics — parts (a)–(g)

Everything below is computed on the regular session and resampled to 1-minute bins.

In [ ]:
def minute_stats(d, avg_spread=None):
    """d = STOCK_DATA[sym]. Returns a per-minute DataFrame with parts (a)-(g)."""
    tr, qt = d["trades"], d["bbo"]
    M = "1min"
    out = pd.DataFrame(index=pd.date_range(SESSION[0], SESSION[1], freq=M))

    # (a) dollar volume, (b) #trades, (c) OHLC, (d) VWAP
    if len(tr):
        g = tr.resample(M)
        out["dollar_vol"] = g["dollar"].sum()
        out["n_trades"]   = g["price"].count()
        out["open"]  = g["price"].first(); out["close"] = g["price"].last()
        out["high"]  = g["price"].max();   out["low"]   = g["price"].min()
        out["vwap"]  = g["dollar"].sum() / g["shares"].sum()

    # (e) BBO spread & depth (time-weighted within the minute via last-obs snapshots)
    if len(qt):
        q = qt.copy(); q["depth"] = q["bidsz"] + q["asksz"]
        gq = q.resample(M)
        out["spread"] = gq["spread"].mean()
        out["depth"]  = gq["depth"].mean()
        # (f) depth counted only while spread <= 2 * day-average spread, else 0
        thr = (2 * q["spread"].mean()) if avg_spread is None else 2 * avg_spread
        q["depth_thr"] = np.where(q["spread"] <= thr, q["depth"], 0.0)
        out["depth_within_2xspread"] = q.resample(M)["depth_thr"].mean()

    # (g) 5-second price impact: mid(t+5s)/mid(t) log-return regressed on trade sign, per minute
    out["price_impact_5s"] = np.nan
    if len(tr) and len(qt):
        mid = qt["mid"].sort_index()
        t0 = tr.index; t5 = t0 + pd.Timedelta(seconds=5)
        pos0 = mid.index.searchsorted(t0, side="right") - 1
        pos5 = mid.index.searchsorted(t5, side="right") - 1
        ok = (pos0 >= 0) & (pos5 >= 0)
        m0 = mid.values[np.clip(pos0, 0, len(mid)-1)]
        m5 = mid.values[np.clip(pos5, 0, len(mid)-1)]
        r5 = np.where(ok & (m0 > 0), np.log(m5 / m0), np.nan)
        pi = pd.DataFrame({"r5": r5, "sign": tr["sign"].values}, index=t0).dropna()
        def _lambda(x):
            if len(x) < 5 or x["sign"].nunique() < 2: return np.nan
            X = sm.add_constant(x["sign"].values)
            try: return sm.OLS(x["r5"].values, X).fit().params[1]
            except Exception: return np.nan
        out["price_impact_5s"] = pi.resample(M).apply(_lambda)
    return out


## 6 · Price series, returns, realized variance, autocorrelation — parts (h)–(k)

* **(h)** last midquote and last trade price in each 1-second and 1-minute bin
* **(i)** log-returns of both series
* **(j)** realized variance $RV=\sum_t r_t^2$
* **(k)** Box–Pierce/Ljung–Box statistic and the first-lag autocorrelation (PACF)

In [ ]:
def price_series(d, freq):
    """(h) last midquote & last trade price per bin -> (i) log-returns."""
    tr, qt = d["trades"], d["bbo"]
    idx = pd.date_range(SESSION[0], SESSION[1], freq=freq)
    mid  = qt["mid"].resample(freq).last().reindex(idx).ffill()  if len(qt) else pd.Series(index=idx, dtype=float)
    trade= tr["price"].resample(freq).last().reindex(idx).ffill() if len(tr) else pd.Series(index=idx, dtype=float)
    df = pd.DataFrame({"mid": mid, "trade": trade})
    df["mid_ret"]   = np.log(df["mid"]).diff()
    df["trade_ret"] = np.log(df["trade"]).diff()
    return df

def realized_variance(ret):                    # (j)
    r = pd.Series(ret).dropna(); return float((r**2).sum())

def return_diagnostics(ret, lags=10):          # (k)
    r = pd.Series(ret).dropna()
    if len(r) < lags + 5:
        return {"RV": realized_variance(r), "ac1": np.nan, "BoxPierce_p": np.nan}
    lb = acorr_ljungbox(r, lags=[lags], boxpierce=True, return_df=True)
    ac1 = acf(r, nlags=1, fft=False)[1]
    return {"RV": realized_variance(r),
            "ac1": float(ac1),
            "BoxPierce_stat": float(lb["bp_stat"].iloc[0]),
            "BoxPierce_p": float(lb["bp_pvalue"].iloc[0])}


## 7 · Assemble the summary-statistics table

In [ ]:
def summarize_instrument(name, d, is_stock=True):
    ms = minute_stats(d)
    row = {"instrument": name}
    def desc(col):
        if col in ms and ms[col].notna().any():
            s = ms[col].dropna()
            return s.mean(), s.std(), s.min(), s.max()
        return (np.nan,)*4
    fields = ["dollar_vol", "n_trades", "vwap", "spread", "depth",
              "depth_within_2xspread", "price_impact_5s"]
    for fld in fields:
        mu, sd, lo, hi = desc(fld)
        row[f"{fld}_mean"] = mu; row[f"{fld}_std"] = sd
        row[f"{fld}_min"]  = lo; row[f"{fld}_max"] = hi
    if is_stock:
        row["n_orders_day"] = d["n_orders"]
    # returns / RV / autocorrelation at 1-second and 1-minute
    for freq, tag in [("1s", "1s"), ("1min", "1min")]:
        ps = price_series(d, freq)
        for which in ["mid_ret", "trade_ret"]:
            diag = return_diagnostics(ps[which], lags=10)
            row[f"RV_{which}_{tag}"]  = diag["RV"]
            row[f"ac1_{which}_{tag}"] = diag["ac1"]
    return row, ms

# SUMMARY, MINUTES = {}, {}
# rows = []
# for s in STOCKS:
#     r, ms = summarize_instrument(s, STOCK_DATA[s], is_stock=True)
#     rows.append(r); MINUTES[s] = ms
# SUMMARY_TABLE = pd.DataFrame(rows).set_index("instrument").T
# SUMMARY_TABLE


## 8 · Triangular arbitrage in the FX triangle — part (l)

Align the three pairs on a 1-second grid (last quote in each second). Using executable bid/ask
quotes, a full round trip that starts and ends in EUR must return **≤ 1 EUR** in the absence of
arbitrage; a value **> 1** (after costs) is a triangular-arbitrage opportunity.

$$\text{Loop A (EUR}\!\to\!\text{USD}\!\to\!\text{JPY}\!\to\!\text{EUR):}\quad
   \frac{\text{EURUSD}_{bid}\cdot \text{USDJPY}_{bid}}{\text{EURJPY}_{ask}}$$
$$\text{Loop B (EUR}\!\to\!\text{JPY}\!\to\!\text{USD}\!\to\!\text{EUR):}\quad
   \frac{\text{EURJPY}_{bid}}{\text{USDJPY}_{ask}\cdot \text{EURUSD}_{ask}}$$

We report **frequency** (seconds with an opportunity), **duration** (length of consecutive runs),
and **tradable quantity** (the binding leg size, converted to EUR).

In [ ]:
def triangular_arbitrage(FX):
    grid = pd.date_range(SESSION[0], SESSION[1], freq="1s")
    def onesec(p):
        q = FX[p][["bid", "ask", "bidsz", "asksz"]] if "bidsz" in FX[p] else FX[p][["bid", "ask"]]
        return q.resample("1s").last().reindex(grid).ffill()
    eu, uj, ej = onesec("EURUSD"), onesec("USDJPY"), onesec("EURJPY")
    df = pd.DataFrame(index=grid)
    df["A"] = eu["bid"] * uj["bid"] / ej["ask"]        # EUR->USD->JPY->EUR
    df["B"] = ej["bid"] / (uj["ask"] * eu["ask"])      # EUR->JPY->USD->EUR
    df["profit_bps"] = (np.maximum(df["A"], df["B"]) - 1.0) * 1e4
    df["arb"] = df["profit_bps"] > 0

    # tradable size (EUR) at the binding leg, when sizes are available
    if "bidsz" in eu:
        sizeA = np.minimum.reduce([eu["bidsz"], uj["bidsz"] / eu["bid"], ej["asksz"]])
        sizeB = np.minimum.reduce([ej["bidsz"], uj["asksz"] / eu["ask"], eu["asksz"]])
        df["qty_eur"] = np.where(df["A"] >= df["B"], sizeA, sizeB)
        df.loc[~df["arb"], "qty_eur"] = 0.0

    arb = df[df["arb"]]
    # durations = lengths of consecutive-second runs
    runs = []
    if len(arb):
        gap = arb.index.to_series().diff().dt.total_seconds().fillna(2) > 1
        grp = gap.cumsum()
        runs = arb.groupby(grp).size().values
    stats = {
        "seconds_with_arb": int(df["arb"].sum()),
        "fraction_of_day": float(df["arb"].mean()),
        "mean_profit_bps": float(arb["profit_bps"].mean()) if len(arb) else 0.0,
        "max_profit_bps":  float(arb["profit_bps"].max())  if len(arb) else 0.0,
        "n_episodes": int(len(runs)),
        "mean_duration_s": float(np.mean(runs)) if len(runs) else 0.0,
        "max_duration_s":  float(np.max(runs))  if len(runs) else 0.0,
    }
    if "qty_eur" in df:
        stats["mean_tradable_eur"] = float(arb["qty_eur"].mean()) if len(arb) else 0.0
    return df, stats

# TRI_DF, TRI_STATS = triangular_arbitrage(FX)
# pd.Series(TRI_STATS)


## 9 · Plots

1. **Intraday liquidity** — spread, depth, and 5-sec price impact averaged over each **5-minute**
   bin (09:30–09:35 … 15:55–16:00), one panel per measure, both stocks overlaid.
2. **Return dynamics** — variance and first-lag autocorrelation of the 1-minute transaction
   log-returns over each **30-minute** bin.
3. **ACF/PACF** of midquote and transaction returns (≈10 lags).
4. **Triangular-arbitrage profit** over the day.

In [ ]:
def bin_label_index(freq):
    edges = pd.date_range(SESSION[0], SESSION[1], freq=freq)
    return [f"{a:%H:%M}-{b:%H:%M}" for a, b in zip(edges[:-1], edges[1:])], edges

def plot_intraday_liquidity(MINUTES):
    labels, edges = bin_label_index("5min")
    for measure, title in [("spread", "BBO spread ($)"),
                           ("depth", "BBO depth (shares)"),
                           ("price_impact_5s", "5-sec price impact (λ)")]:
        fig, ax = plt.subplots()
        for s in STOCKS:
            ms = MINUTES[s]
            if measure not in ms: continue
            b = ms[measure].resample("5min").mean().reindex(edges[:-1])
            ax.plot(range(len(b)), b.values, marker=".", label=s)
        ax.set_xticks(range(0, len(labels), 6))
        ax.set_xticklabels([labels[i] for i in range(0, len(labels), 6)], rotation=45, ha="right")
        ax.set_title(f"Intraday {title} — 5-min bins ({DATE})"); ax.legend(); ax.grid(alpha=.3)
        plt.tight_layout(); plt.show()

def plot_return_dynamics(d, name):
    ps = price_series(d, "1min")
    labels, edges = bin_label_index("30min")
    var, ac1 = [], []
    for a, b in zip(edges[:-1], edges[1:]):
        r = ps["trade_ret"].loc[(ps.index > a) & (ps.index <= b)].dropna()
        var.append((r**2).sum())
        ac1.append(acf(r, nlags=1, fft=False)[1] if len(r) > 3 else np.nan)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].bar(range(len(var)), var); ax[0].set_title(f"{name}: 1-min trade-return variance / 30-min")
    ax[1].bar(range(len(ac1)), ac1); ax[1].set_title(f"{name}: first-lag autocorrelation / 30-min")
    for a in ax:
        a.set_xticks(range(len(labels))); a.set_xticklabels(labels, rotation=45, ha="right"); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()

def plot_acf(d, name, freq="1min", lags=10):
    ps = price_series(d, freq)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for j, which in enumerate(["mid_ret", "trade_ret"]):
        r = ps[which].dropna()
        a = acf(r, nlags=lags, fft=False)
        ax[j].bar(range(len(a)), a)
        ci = 1.96 / np.sqrt(len(r))
        ax[j].axhline(ci, ls="--", c="r"); ax[j].axhline(-ci, ls="--", c="r")
        ax[j].set_title(f"{name} {which} ACF ({freq})"); ax[j].grid(alpha=.3)
    plt.tight_layout(); plt.show()

# plot_intraday_liquidity(MINUTES)
# for s in STOCKS: plot_return_dynamics(STOCK_DATA[s], s); plot_acf(STOCK_DATA[s], s)
# TRI_DF["profit_bps"].clip(lower=0).plot(title="Triangular-arbitrage profit (bps), 1-sec"); plt.show()


## 10 · Run everything

In [ ]:
# ----- full pipeline (uncomment to execute end-to-end) -----
# download_itch()
# STOCK_DATA = parse_itch(STOCKS)
# FX = load_fx()
#
# rows, MINUTES = [], {}
# for s in STOCKS:
#     r, ms = summarize_instrument(s, STOCK_DATA[s], is_stock=True); rows.append(r); MINUTES[s] = ms
# SUMMARY_TABLE = pd.DataFrame(rows).set_index("instrument").T
# SUMMARY_TABLE.to_csv("summary_statistics.csv")
# display(SUMMARY_TABLE)
#
# TRI_DF, TRI_STATS = triangular_arbitrage(FX); print(pd.Series(TRI_STATS))
#
# plot_intraday_liquidity(MINUTES)
# for s in STOCKS:
#     plot_return_dynamics(STOCK_DATA[s], s); plot_acf(STOCK_DATA[s], s)
# TRI_DF["profit_bps"].clip(lower=0).plot(title="Triangular-arbitrage profit (bps)"); plt.show()


## 11 · Report — observations & economics

*Fill the bracketed numbers from `SUMMARY_TABLE` / `TRI_STATS` after running the pipeline.*

**Size, spread and depth (parts a–f).** AAPL trades vastly more dollar volume per minute than
GPRO, has a spread that is a small **[…]** bps and typically pinned near the one-cent minimum tick,
and carries much larger displayed depth. GPRO's spread is wider and more volatile and its depth
thinner. This is the classic cross-section of liquidity: large, information-rich, heavily
intermediated stocks have low adverse-selection costs and tight, deep markets, while small caps
have higher inventory and adverse-selection risk, so market makers quote wider and shallower. The
"depth within 2× average spread" measure (f) collapses to zero for GPRO in exactly the minutes when
the spread blows out — those are precisely the moments liquidity demanders face the worst terms.

**Price impact (part g).** The 5-second impact coefficient λ is markedly larger for GPRO than for
AAPL: a signed trade moves the small-cap midquote far more, because each trade carries a larger
share of the total information and meets less standing depth (Kyle's λ ∝ information / depth). λ
also rises near the open and around news, tracing the intraday liquidity U-shape.

**Return autocorrelation (parts i–k).** High-frequency **transaction** returns show pronounced
**negative** first-order autocorrelation — the mechanical signature of bid-ask bounce (Roll):
trades alternate between bid and ask, so price changes reverse. **Midquote** returns show much
weaker autocorrelation because the midpoint filters out the bounce; what remains is closer to a
martingale, consistent with efficient prices. The effect is stronger and the Box–Pierce statistic
larger for GPRO, whose wider spread makes the bounce mechanically bigger. Realized variance is
higher at the 1-second than the per-trade-implied level for the transaction series and shrinks for
the midquote series — again the bounce inflating measured transaction volatility (the classic
motivation for using midquotes / realized-variance corrections).

**Triangular arbitrage (part l).** Opportunities in the EUR/USD–USD/JPY–EUR/JPY triangle are
**rare, tiny, and extremely short-lived**: they appear in only **[…]%** of seconds, average
**[…]** bps, and last a second or two before quotes re-align. Once realistic bid/ask costs are
imposed most apparent gross deviations vanish. This is the FX market's efficiency at work — the
cross rate is arbitraged against its two legs almost instantly — and the tradable size at the
binding leg is what caps any actual profit.

**Tie-back.** Both markets illustrate the course's central theme: prices are efficient at the
midquote but observed transaction prices are distorted by the market microstructure (discrete
ticks, bid-ask bounce, finite depth), and the *magnitude* of those distortions scales inversely
with liquidity — largest for the small-cap stock, smallest in deep FX.
